# Incident Response Runbook: Discovery - System Information Discovery

**Tactic:** Discovery
**Technique:** System Information Discovery (T1082)
**Severity:** MEDIUM

## Overview

This runbook provides step-by-step incident response procedures for System Information Discovery activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for system information discovery indicators
print(f"\n[QUERY] Searching Splunk for system information discovery indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=4688)
OR (sourcetype=linux_secure "systeminfo" OR "hostname" OR "whoami" OR "uname" OR "lsb_release")
OR (sourcetype=network "SMB" OR "WMI" OR "RPC")
| eval command=coalesce(CommandLine, cmd, command)
| where match(command, "systeminfo|hostname|whoami|uname|lsb_release|wmic.*computersystem|Get-ComputerInfo|Get-WmiObject.*Win32_ComputerSystem")
| stats count by host, user, command, _time
| where count > 5
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential system information discovery events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and suspicious activities
affected_systems = []
suspicious_activities = []
splunk_indicators = []
unique_users = set()

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'user': event.get('user', 'unknown'),
        'command': event.get('command', ''),
        'event_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    unique_users.add(event.get('user', 'unknown'))

    # Extract indicators
    if event.get('command'):
        splunk_indicators.append({
            'type': 'system_discovery_command',
            'value': event.get('command'),
            'context': f"Executed by {event.get('user', 'unknown')} on {event.get('host', 'unknown')}"
        })

# Query CrowdStrike for endpoint detection
print(f"\n[QUERY] Checking CrowdStrike for system information discovery detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'System Information Discovery'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0)
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            indicator['threat_intel'] = misp_search[0] if misp_search else None
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'System Information Discovery Incident - {len(affected_systems)} systems',
        'description': f'Detected system information discovery activities on {len(affected_systems)} systems',
        'severity': 'MEDIUM',
        'tactic': 'Discovery',
        'technique': 'System Information Discovery (T1082)',
        'indicators': splunk_indicators,
        'affected_systems': affected_systems,
        'threat_intelligence': misp_results
    }

    incident_id = iris.create_case(incident_data)
    if incident_id:
        print(f"   Created IRIS case: {incident_id}")
    else:
        print(f"   Failed to create IRIS case")
        incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - Affected systems: {len(affected_systems)}")
print(f"  - Unique users: {len(unique_users)}")
print(f"  - Suspicious indicators: {len(splunk_indicators)}")
print(f"  - Threat intelligence hits: {len(misp_results)}")
print(f"  - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
disabled_accounts = []
blocked_ips = []

# 1. Isolate affected systems
print(f"\n[CONTAINMENT] Isolating affected systems...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Use CrowdStrike to isolate host
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'host_isolation',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Isolated host: {system['hostname']}")
            else:
                print(f"   Failed to isolate host: {system['hostname']}")
        else:
            # Use Shuffle for network isolation
            network_isolation = shuffle.isolate_system(system['hostname'])
            if network_isolation:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'network_isolation',
                    'target': system['hostname'],
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Network isolated: {system['hostname']}")
except Exception as e:
    print(f"   Error during host isolation: {e}")

# 2. Disable suspicious accounts
print(f"\n[CONTAINMENT] Disabling suspicious accounts...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Disable account via Shuffle
            disable_result = shuffle.disable_account(user)
            if disable_result:
                disabled_accounts.append(user)
                containment_actions.append({
                    'action': 'account_disable',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Disabled account: {user}")
except Exception as e:
    print(f"   Error disabling accounts: {e}")

# 3. Block suspicious IPs/domains
print(f"\n[CONTAINMENT] Blocking suspicious IPs and domains...")
try:
    suspicious_ips = []
    for indicator in splunk_indicators:
        # Extract potential IPs from commands or context
        ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
        ips = re.findall(ip_pattern, indicator.get('context', ''))
        suspicious_ips.extend(ips)

    for ip in set(suspicious_ips):
        block_result = shuffle.block_ip(ip)
        if block_result:
            blocked_ips.append(ip)
            containment_actions.append({
                'action': 'ip_block',
                'target': ip,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': containment_time
            })
            print(f"   Blocked IP: {ip}")
except Exception as e:
    print(f"   Error blocking IPs: {e}")

# 4. Enable enhanced monitoring
print(f"\n[CONTAINMENT] Enabling enhanced monitoring...")
try:
    monitoring_rules = [
        {
            'name': 'Enhanced System Discovery Monitoring',
            'query': 'index=* EventCode=4688 | eval risk_score = case(match(CommandLine, "systeminfo|hostname|whoami|uname"), 8, match(CommandLine, "wmic.*computersystem|Get-ComputerInfo"), 9, 1==1, 4) | where risk_score >= 7',
            'alert_threshold': 3,
            'time_window': '10m'
        }
    ]

    enhanced_monitoring = splunk.enable_enhanced_monitoring(monitoring_rules)
    if enhanced_monitoring:
        containment_actions.append({
            'action': 'enhanced_monitoring',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enabled enhanced monitoring in Splunk")
except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

print(f"\n Containment complete:")
print(f"  - Hosts isolated: {len(isolated_hosts)}")
print(f"  - Accounts disabled: {len(disabled_accounts)}")
print(f"  - IPs blocked: {len(blocked_ips)}")
print(f"  - Enhanced monitoring: enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
terminated_processes = []
deleted_tools = []
reset_credentials = []

# 1. Terminate suspicious processes
print(f"\n[ERADICATION] Terminating suspicious processes...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Get running processes from CrowdStrike
            processes = crowdstrike.get_processes(system['device_id'])
            suspicious_procs = []

            for proc in processes:
                cmd_line = proc.get('command_line', '').lower()
                if any(keyword in cmd_line for keyword in ['systeminfo', 'hostname', 'whoami', 'uname', 'wmic', 'get-computerinfo']):
                    suspicious_procs.append(proc)

            for proc in suspicious_procs:
                kill_result = crowdstrike.kill_process(system['device_id'], proc['process_id'])
                if kill_result:
                    terminated_processes.append(f"{system['hostname']}:{proc['process_name']}")
                    eradication_actions.append({
                        'action': 'process_termination',
                        'target': f"{system['hostname']}:{proc['process_name']}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Terminated process: {proc['process_name']} on {system['hostname']}")
except Exception as e:
    print(f"   Error terminating processes: {e}")

# 2. Remove discovery tools and scripts
print(f"\n[ERADICATION] Removing discovery tools and scripts...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Scan for and remove common discovery tools
            tools_to_remove = [
                'systeminfo.exe', 'hostname.exe', 'whoami.exe',
                'reconnaissance.ps1', 'discovery.bat', 'enum.vbs'
            ]

            for tool in tools_to_remove:
                removal_result = crowdstrike.remove_file(system['device_id'], tool)
                if removal_result:
                    deleted_tools.append(f"{system['hostname']}:{tool}")
                    eradication_actions.append({
                        'action': 'tool_removal',
                        'target': f"{system['hostname']}:{tool}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Removed tool: {tool} from {system['hostname']}")
except Exception as e:
    print(f"   Error removing tools: {e}")

# 3. Reset compromised credentials
print(f"\n[ERADICATION] Resetting compromised credentials...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Reset password via Shuffle
            reset_result = shuffle.reset_password(user)
            if reset_result:
                reset_credentials.append(user)
                eradication_actions.append({
                    'action': 'credential_reset',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Reset credentials for: {user}")
except Exception as e:
    print(f"   Error resetting credentials: {e}")

# 4. Clean up persistence mechanisms
print(f"\n[ERADICATION] Cleaning up persistence mechanisms...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Remove scheduled tasks or registry entries related to discovery
            persistence_cleanup = crowdstrike.cleanup_persistence(system['device_id'], 'discovery')
            if persistence_cleanup:
                eradication_actions.append({
                    'action': 'persistence_cleanup',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Cleaned persistence on: {system['hostname']}")
except Exception as e:
    print(f"   Error cleaning persistence: {e}")

# 5. Verify eradication
print(f"\n[ERADICATION] Verifying eradication...")
try:
    verification_results = []
    for system in affected_systems:
        if system.get('device_id'):
            # Run verification scan
            scan_result = crowdstrike.scan_system(system['device_id'], 'discovery_indicators')
            verification_results.append({
                'system': system['hostname'],
                'scan_result': scan_result,
                'clean': scan_result.get('threats_found', 0) == 0
            })

    clean_systems = [r for r in verification_results if r['clean']]
    eradication_actions.append({
        'action': 'eradication_verification',
        'target': f"{len(clean_systems)}/{len(verification_results)} systems clean",
        'method': 'CrowdStrike',
        'status': 'success' if len(clean_systems) == len(verification_results) else 'partial',
        'timestamp': eradication_time
    })

    print(f"   Verification complete: {len(clean_systems)}/{len(verification_results)} systems clean")

except Exception as e:
    print(f"   Error during verification: {e}")

print(f"\n Eradication complete:")
print(f"  - Processes terminated: {len(terminated_processes)}")
print(f"  - Tools deleted: {len(deleted_tools)}")
print(f"  - Credentials reset: {len(reset_credentials)}")
print(f"  - Systems verified clean: {len([r for r in verification_results if r.get('clean', False)])}")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_accounts = []
restored_services = []

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
try:
    for host in isolated_hosts:
        system = next((s for s in affected_systems if s['hostname'] == host), None)
        if system and system.get('device_id'):
            # Use CrowdStrike to re-enable host
            reenable_result = crowdstrike.reenable_host(system['device_id'])
            if reenable_result:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'host_reenable',
                    'target': host,
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Re-enabled host: {host}")
            else:
                print(f"   Failed to re-enable host: {host}")
        else:
            # Use Shuffle for network re-enablement
            network_restore = shuffle.restore_system(host)
            if network_restore:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'network_restore',
                    'target': host,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Restored network access: {host}")
except Exception as e:
    print(f"   Error re-enabling systems: {e}")

# 2. Restore disabled accounts
print(f"\n[RECOVERY] Restoring disabled accounts...")
try:
    for user in disabled_accounts:
        # Restore account access via Shuffle
        restore_result = shuffle.restore_account(user)
        if restore_result:
            restored_accounts.append(user)
            recovery_actions.append({
                'action': 'account_restore',
                'target': user,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored account: {user}")
except Exception as e:
    print(f"   Error restoring accounts: {e}")

# 3. Restore system functionality
print(f"\n[RECOVERY] Restoring system functionality...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Validate system health
            health_check = crowdstrike.validate_system_health(system['device_id'])
            if health_check and health_check.get('status') == 'healthy':
                recovery_actions.append({
                    'action': 'system_validation',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   System health validated: {system['hostname']}")
            else:
                print(f"   System health issues detected: {system['hostname']}")
except Exception as e:
    print(f"   Error validating system health: {e}")

# 4. Restore monitoring and alerting
print(f"\n[RECOVERY] Restoring monitoring and alerting...")
try:
    # Restore normal Splunk monitoring rules
    normal_monitoring = splunk.restore_normal_monitoring()
    if normal_monitoring:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring in Splunk")

    # Restore CrowdStrike normal operations
    cs_normal_ops = crowdstrike.restore_normal_operations()
    if cs_normal_ops:
        recovery_actions.append({
            'action': 'cs_normal_ops',
            'target': 'CrowdStrike',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal CrowdStrike operations")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 5. Validate recovery
print(f"\n[RECOVERY] Validating recovery...")
try:
    validation_results = []
    for system in affected_systems:
        # Test basic system functionality
        test_result = shuffle.test_system_functionality(system['hostname'])
        validation_results.append({
            'system': system['hostname'],
            'functional': test_result.get('success', False),
            'services': test_result.get('services_running', [])
        })

    functional_systems = [r for r in validation_results if r['functional']]
    recovery_actions.append({
        'action': 'recovery_validation',
        'target': f"{len(functional_systems)}/{len(validation_results)} systems functional",
        'method': 'Shuffle',
        'status': 'success' if len(functional_systems) == len(validation_results) else 'partial',
        'timestamp': recovery_time
    })

    print(f"   Validation complete: {len(functional_systems)}/{len(validation_results)} systems functional")

except Exception as e:
    print(f"   Error during validation: {e}")

print(f"\n Recovery complete:")
print(f"  - Hosts re-enabled: {len(reenabled_hosts)}")
print(f"  - Accounts restored: {len(restored_accounts)}")
print(f"  - Systems validated: {len([r for r in validation_results if r.get('functional', False)])}")
print(f"  - Monitoring restored: ")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': f'System Information Discovery Incident Report - {len(affected_systems)} systems',
        'detection_time': detection_time,
        'closure_time': closure_time,
        'severity': 'MEDIUM',
        'tactic': 'Discovery',
        'technique': 'System Information Discovery (T1082)',
        'summary': {
            'affected_users': len(unique_users),
            'affected_systems': len(affected_systems),
            'indicators_detected': len(splunk_indicators),
            'hosts_isolated': len(isolated_hosts),
            'accounts_disabled': len(disabled_accounts),
            'processes_terminated': len(terminated_processes),
            'tools_deleted': len(deleted_tools),
            'credentials_reset': len(reset_credentials)
        },
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': closure_time
        },
        'actions_taken': {
            'detection': splunk_indicators[:10],  # Top 10 indicators
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'threat_intelligence': [i.get('threat_intel') for i in splunk_indicators if i.get('threat_intel')],
        'recommendations': [
            'Implement system discovery monitoring and alerting',
            'Restrict access to system enumeration commands',
            'Deploy endpoint detection for reconnaissance tools',
            'Regular security awareness training on discovery detection',
            'Enhance logging for system information gathering activities'
        ]
    }

    # Save report to file
    report_filename = f"system_discovery_report_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'report_generation',
        'target': report_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Generated incident report: {report_filename}")

except Exception as e:
    print(f"   Error generating report: {e}")

# 2. Document lessons learned
print(f"\n[POST-INCIDENT] Documenting lessons learned...")
try:
    lessons_learned = {
        'incident_id': incident_id,
        'what_went_well': [
            'Rapid detection of system information discovery activities',
            'Effective isolation of affected systems',
            'Comprehensive eradication of discovery tools',
            'Successful system recovery and validation'
        ],
        'what_could_improve': [
            'Earlier detection of reconnaissance patterns',
            'Enhanced monitoring of system enumeration',
            'Automated blocking of discovery commands',
            'Better user training on suspicious activity reporting'
        ],
        'key_findings': [
            f'System discovery affected {len(unique_users)} users across {len(affected_systems)} systems',
            f'Most common discovery method: {max([i.get("type", "unknown") for i in splunk_indicators], key=[i.get("type", "unknown") for i in splunk_indicators].count)}',
            'Threat intelligence enriched detection for key reconnaissance tools',
            'Automated response prevented further lateral movement'
        ],
        'preventive_measures': [
            'Implement command-line auditing and restrictions',
            'Deploy advanced endpoint protection with behavior monitoring',
            'Regular vulnerability scanning and system hardening',
            'Enhanced security monitoring and alerting',
            'Security awareness training focused on reconnaissance detection'
        ]
    }

    # Save lessons learned
    lessons_filename = f"system_discovery_lessons_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(lessons_filename, 'w') as f:
        json.dump(lessons_learned, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'lessons_learned',
        'target': lessons_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Documented lessons learned: {lessons_filename}")

except Exception as e:
    print(f"   Error documenting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_measures = []

    # Update Splunk correlation rules for system discovery
    updated_rules = splunk.update_correlation_rules([
        {
            'name': 'Enhanced System Discovery Detection',
            'search': 'index=* sourcetype=WinEventLog:Security EventCode=4688 | eval risk_score = case(match(CommandLine, "systeminfo|hostname|whoami|uname"), 8, match(CommandLine, "wmic.*computersystem|Get-ComputerInfo"), 9, 1==1, 4) | where risk_score >= 7',
            'alert_threshold': 3,
            'time_window': '10m'
        }
    ])
    if updated_rules:
        preventive_measures.append('Updated Splunk system discovery rules')
        print(f"   Updated Splunk correlation rules for system discovery detection")

    # Enhance CrowdStrike prevention policies
    cs_policies = crowdstrike.update_prevention_policies({
        'system_discovery_prevention': 'enabled',
        'command_line_auditing': 'strict',
        'reconnaissance_tool_detection': 'enabled',
        'system_enumeration_monitoring': 'enabled'
    })
    if cs_policies:
        preventive_measures.append('Enhanced CrowdStrike discovery policies')
        print(f"   Enhanced CrowdStrike prevention policies")

    # Schedule system discovery security training
    training_scheduled = shuffle.schedule_security_training(
        users=list(unique_users),
        topic='Detecting and Reporting System Discovery Activities',
        incident_reference=incident_id
    )
    if training_scheduled:
        preventive_measures.append('Scheduled reconnaissance training')
        print(f"   Scheduled security awareness training on system discovery detection")

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Share threat intelligence
print(f"\n[POST-INCIDENT] Sharing threat intelligence...")
try:
    shared_intel = []
    for indicator in splunk_indicators:
        if indicator.get('threat_intel'):
            # Share with MISP
            result = misp.share_indicator(indicator, incident_id)
            if result:
                shared_intel.append(f"MISP: {indicator.get('type', 'unknown')}")
                print(f"   Shared indicator with MISP: {indicator.get('type', 'unknown')}")

    if shared_intel:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': shared_intel,
            'status': 'success',
            'timestamp': closure_time
        })

except Exception as e:
    print(f"   Error sharing threat intelligence: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    if incident_id:
        closure_data = {
            'status': 'closed',
            'closure_time': closure_time,
            'closure_reason': 'System information discovery incident successfully contained, eradicated, and recovered',
            'post_incident_actions': post_incident_actions,
            'final_assessment': {
                'threat_contained': len(isolated_hosts) > 0,
                'threat_eradicated': len(terminated_processes) > 0 or len(deleted_tools) > 0,
                'systems_recovered': len(reenabled_hosts) > 0,
                'preventive_measures': len(preventive_measures) > 0
            }
        }

        result = iris.close_case(incident_id, closure_data)
        if result:
            post_incident_actions.append({
                'action': 'case_closure',
                'target': incident_id,
                'status': 'success',
                'timestamp': closure_time
            })
            print(f"   Closed IRIS case: {incident_id}")
        else:
            print(f"   Failed to close IRIS case: {incident_id}")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated")
print(f"  - Lessons learned documented")
print(f"  - {len(preventive_measures)} preventive measures implemented")
print(f"  - Threat intelligence shared with {len(shared_intel)} platforms")
print(f"  - Incident case closed: {incident_id}")

print(f"\n System Information Discovery Incident Response Complete")
print(f"   Total duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")
print(f"   Response effectiveness: {'HIGH' if len(isolated_hosts) > 0 and len(terminated_processes) > 0 else 'MEDIUM'}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
